## This notebook performs data attribution for the Kellect dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

In [ ]:
from pprint import pprint
import gzip
from sklearn.manifold import TSNE
import json
import copy
import os

import gensim
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

import multiprocessing

import os
import json
import pickle

import torch
import numpy as np
from torch.nn import CrossEntropyLoss
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric import utils
from sklearn.utils import class_weight

In [ ]:
def get_json_paths(root_dir):
    json_paths = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower().endswith('.json'):
                full_path = os.path.join(dirpath, fname)
                json_paths.append(full_path)
    return json_paths

root = "kellect/public_data"
paths = get_json_paths(root)
paths.sort()

In [ ]:
import json

def read_json_records(paths):
    records = []
    for path in paths:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:  # skip empty lines
                    records.append(json.loads(line))
    return records

In [ ]:
records = read_json_records(paths[30:60])
malicious_records = read_json_records(paths[:30])

In [ ]:
def construct_df(records):
    df = pd.DataFrame.from_records(records)
    events = ['FileIORead','FileIOWrite','ProcessStart','TcpIpRecvIPV4','TcpIpSendIPV4']
    df = df[df['Event'].isin(events)]
    df[['objectname', 'objectID', 'actorname', 'actorID', 'object']] = ''

    df.loc[df['Event']=='TcpIpSendIPV4', 'objectname'] = \
        df.loc[df['Event']=='TcpIpSendIPV4', 'args'].str['daddr']
    df.loc[df['Event']=='TcpIpRecvIPV4', 'objectname'] = \
        df.loc[df['Event']=='TcpIpRecvIPV4', 'args'].str['daddr']
    df.loc[df['Event']=='FileIORead',    'objectname'] = \
        df.loc[df['Event']=='FileIORead',    'args'].str['FileName']
    df.loc[df['Event']=='FileIOWrite',   'objectname'] = \
        df.loc[df['Event']=='FileIOWrite',   'args'].str['FileName']
    df.loc[df['Event']=='ProcessStart',  'objectname'] = \
        df.loc[df['Event']=='ProcessStart',  'args'].str['ImageFileName']
    
    df.loc[df['Event']=='TcpIpSendIPV4', 'objectID'] = \
        df.loc[df['Event']=='TcpIpSendIPV4', 'args'].str['daddr']
    df.loc[df['Event']=='TcpIpRecvIPV4', 'objectID'] = \
        df.loc[df['Event']=='TcpIpRecvIPV4', 'args'].str['daddr']
    df.loc[df['Event']=='FileIORead',    'objectID'] = \
        df.loc[df['Event']=='FileIORead',    'args'].str['FileName']
    df.loc[df['Event']=='FileIOWrite',   'objectID'] = \
        df.loc[df['Event']=='FileIOWrite',   'args'].str['FileName']
    df.loc[df['Event']=='ProcessStart',  'objectID'] = \
        df.loc[df['Event']=='ProcessStart',  'args'].str['ProcessId']
    
    df.loc[df['Event']=='TcpIpSendIPV4', 'actorID'] = \
        df.loc[df['Event']=='TcpIpSendIPV4', 'PID']
    df.loc[df['Event']=='TcpIpRecvIPV4', 'actorID'] = \
        df.loc[df['Event']=='TcpIpRecvIPV4', 'PID']
    df.loc[df['Event']=='FileIORead',    'actorID'] = \
        df.loc[df['Event']=='FileIORead',    'PID']
    df.loc[df['Event']=='FileIOWrite',   'actorID'] = \
        df.loc[df['Event']=='FileIOWrite',   'PID']
    df.loc[df['Event']=='ProcessStart',  'actorID'] = \
        df.loc[df['Event']=='ProcessStart',  'PPID']
    
    df.loc[df['Event']=='TcpIpSendIPV4', 'actorname'] = \
        df.loc[df['Event']=='TcpIpSendIPV4', 'PName']
    df.loc[df['Event']=='TcpIpRecvIPV4', 'actorname'] = \
        df.loc[df['Event']=='TcpIpRecvIPV4', 'PName']
    df.loc[df['Event']=='FileIORead',    'actorname'] = \
        df.loc[df['Event']=='FileIORead',    'PName']
    df.loc[df['Event']=='FileIOWrite',   'actorname'] = \
        df.loc[df['Event']=='FileIOWrite',   'PName']
    df.loc[df['Event']=='ProcessStart',  'actorname'] = \
        df.loc[df['Event']=='ProcessStart',  'PName']
    
    df.loc[df['Event']=='TcpIpSendIPV4', 'object'] = 'FLOW'
    df.loc[df['Event']=='TcpIpRecvIPV4', 'object'] = 'FLOW'
    df.loc[df['Event']=='FileIORead',    'object'] = 'FILE'
    df.loc[df['Event']=='FileIOWrite',   'object'] = 'FILE'
    df.loc[df['Event']=='ProcessStart',  'object'] = 'PROCESS'
    
    df = df[['Event','actorID','objectID','object','actorname','objectname','TimeStamp']]
    df = df.rename(columns={'Event': 'action'})
    
    return df

In [ ]:
def preprocess(df):
    df = df[df['actorID'] != df['objectID']]
    df = df.drop_duplicates(
        subset=['action','actorID','objectID','object','actorname','objectname'],
        keep='first'
    )
    return df

In [ ]:
def prepare_graph(df):
    nodes = {}
    labels = {}
    edges = []

    dummies = {'PROCESS': 0, 'FLOW': 1, 'FILE': 2}
    lblmap = {}

    for i in range(len(df)):
        x = df.iloc[i]
        actorid = x['actorID']
        objectid = x['objectID']

        if actorid not in nodes:
            nodes[actorid] = []
        if objectid not in nodes:
            nodes[objectid] = []

        nodes[actorid] += [x['actorname'], x['action'], x['objectname']]
        nodes[objectid] += [x['actorname'], x['action'], x['objectname']]

        labels[actorid] = dummies['PROCESS']
        labels[objectid] = dummies[x['object']]

        lblmap[actorid] = x['actorname']
        lblmap[objectid] = x['objectname']

        edges.append((actorid, objectid))

    features = []
    feat_labels = []
    edge_index = [[], []]
    index = {}
    mapp = []

    for k, v in nodes.items():
        features.append(v)
        feat_labels.append(labels[k])
        index[k] = len(features) - 1
        mapp.append(k)

    for src_id, dst_id in edges:
        src = index[src_id]
        dst = index[dst_id]
        edge_index[0].append(src)
        edge_index[1].append(dst)

    return features, feat_labels, edge_index, mapp, lblmap


In [ ]:
from gensim.models.callbacks import CallbackAny2Vec
from gensim.models import Word2Vec

class EpochSaver(CallbackAny2Vec):
    '''Callback to save model after each epoch.'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        model.save('flash_kellect.model')
        self.epoch += 1

In [ ]:
class EpochLogger(CallbackAny2Vec):
    '''Callback to log information about training'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print("Epoch #{} start".format(self.epoch))

    def on_epoch_end(self, model):
        print("Epoch #{} end".format(self.epoch))
        self.epoch += 1

In [ ]:
df = construct_df(records)
df = preprocess(df)

In [ ]:
logger = EpochLogger()
saver = EpochSaver()
word2vec = Word2Vec(sentences=phrases, vector_size=20, window=5, min_count=1, workers=4,epochs=50,callbacks=[saver,logger])

In [ ]:
import torch
import math

class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]


def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(30)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(20)
w2vmodel = Word2Vec.load("flash_kellect.model")

In [ ]:
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
import torch.nn as nn

class GCN(nn.Module):
    def __init__(self, in_channels=20, hidden_channels=32, num_classes=3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, normalize=True)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, normalize=True)
        self.lin   = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.lin(x)
        return F.softmax(x, dim=1)

In [ ]:
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

In [ ]:
import os
import json
import pickle
import time

import torch
import numpy as np
from torch.nn import CrossEntropyLoss
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric import utils
from sklearn.utils import class_weight

start = time.time()

candidates = list(set(df['actorname']))

all_score_matrices = {}
target = 'zeroth'

while len(candidates) > 0:
    print("Target:", target)
    
    #---------------------------------------------------------------------------
    #                           TRAINING SECTION
    #---------------------------------------------------------------------------
    
    # Filter benign data to exclude the current target
    df_trim = df[df["actorname"] != target]

    # Prepare the graph
    phrases, labels ,edges, mapp, lblmapp = prepare_graph(df_trim)    

    model = GCN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    
    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)
    
    criterion = CrossEntropyLoss()
    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device),
        edge_index=torch.tensor(edges, dtype=torch.long).to(device)
    )
    
    for epochs in range(100):
        model.train()
        optimizer.zero_grad() 
        out = model(graph.x, graph.edge_index) 
        loss = criterion(out, graph.y) 
        loss.backward() 
        optimizer.step()      
        total_loss = loss.item() 
    
    torch.save(model.state_dict(), 'flash_kellect.pth')
    #---------------------------------------------------------------------------
    #                           EVALUATION SECTION
    #---------------------------------------------------------------------------
    mode = "evaluation"
    
    # --- CACHING CHANGES ---
    eval_cache_file = "evaluation_cache_kellect.pkl"
    if os.path.exists(eval_cache_file):
        # If cache file already exists, load from it
        with open(eval_cache_file, "rb") as f:
            eval_cache = pickle.load(f)

        all_ids = eval_cache["all_ids"]
        phrases = eval_cache["phrases"]
        labels = eval_cache["labels"]
        edges = eval_cache["edges"]
        mapp = eval_cache["mapp"]
        lbl = eval_cache["lbl"]
        nodes = eval_cache["nodes"]
        gt_nodes = eval_cache["gt_nodes"]

        print("Loaded evaluation data from cache.")
    else:
        # Otherwise, read raw data, transform, and create the graph
        testdf = construct_df(malicious_records)
        testdf = preprocess(testdf)
        phrases, labels ,edges,mapp,lblmap = prepare_graph(testdf)        
        
        nodes = [infer(x) for x in phrases]
        nodes = np.array(nodes)  
        
        all_ids = list(testdf['actorID']) + list(testdf['objectID'])
        all_ids = set(all_ids)
        
        gt_nodes = set()
        for i in range(len(mapp)):
            if labels[i] == 0:
                gt_nodes.add(mapp[i])
        gt_nodes = list(gt_nodes)        

        # Save data to cache
        eval_cache = {
            "all_ids": all_ids,
            "phrases": phrases,
            "labels": labels,
            "edges": edges,
            "mapp": mapp,
            "lbl": lblmap,
            "nodes": nodes,
            "gt_nodes": gt_nodes,
        }
        with open(eval_cache_file, "wb") as f:
            pickle.dump(eval_cache, f)
        print("Evaluation data cached to disk.")

    graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
    
    model.load_state_dict(torch.load(f'flash_kellect.pth'))
    model.eval()
    out = model(graph.x, graph.edge_index)
    
    sorted, indices = out.sort(dim=1, descending=True)
    conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
    conf = (conf - conf.min()) / conf.max()
    scores = conf
    
    score_matrix = {}
    for x in gt_nodes:
        ind = mapp.index(x)
        # For demonstration, store the score under x's label
        score_matrix[lbl[x]] = float(scores[ind].item())

    # Store scores for this target in the global dictionary
    all_score_matrices[target] = score_matrix

    # Save all_score_matrices at the end of this iteration
    with open("all_score_matrices_kellect.pkl", "wb") as f:
        pickle.dump(all_score_matrices, f)

    # Move to the next target
    target = candidates.pop()

print("Done with all targets. Final all_score_matrices saved.")

end = time.time()

In [ ]:
def find_top_k_contributing_samples(all_score_matrices, k=3):
    baseline_scores = all_score_matrices['zeroth']
    top_k_contributions = {}

    for malicious_proc, baseline_score in baseline_scores.items():
        diff_list = []

        for omitted_sample, score_dict in all_score_matrices.items():
            if omitted_sample == 'zeroth':
                continue

            current_score = score_dict.get(malicious_proc)
            if current_score is None:
                continue

            diff_from_baseline = current_score - baseline_score

            diff_list.append((omitted_sample, diff_from_baseline))

        diff_list.sort(key=lambda x: x[1], reverse=True)

        top_k_contributions[malicious_proc] = diff_list[:k]

    return top_k_contributions

In [ ]:
import pickle

with open("all_score_matrices_kellect.pkl", "rb") as f:
    all_score_matrices = pickle.load(f)

# Attribution Results

In [ ]:
results = find_top_k_contributing_samples(all_score_matrices, k=10)
for malicious_proc, top_list in results.items():
    print(f"Malicious process: {malicious_proc}")
    for (omitted_sample, diff) in top_list:
        print(f"  Attributed sample: {omitted_sample}, Score difference: {diff:.4f}")
    print()

In [ ]:
records = read_json_records(paths[30:60])
malicious_records = read_json_records(paths[:30])

df = construct_df(records)
train_df = preprocess(df)

df = construct_df(malicious_records)
test_df = preprocess(df)

In [ ]:
results = find_top_k_contributing_samples(all_score_matrices, k=1)
for malicious_proc, top_list in results.items():
    for (omitted_sample, diff) in top_list:
        print('Alert Entity :', malicious_proc)
        
        target = malicious_proc
        utestdf = test_df[test_df['actorname'] == target][['actorname', 'objectname']]
        utestinetd = utestdf.drop_duplicates()
        count = len(utestinetd)
        print('Evaluation Graph: ',count)
        
        target = omitted_sample
        inetd_df = train_df[train_df['actorname'] == target][['actorname', 'objectname']]
        unique_inetd = inetd_df.drop_duplicates()
        count = len(unique_inetd)
        print('Attribution Graph: ', count)

        merged = utestinetd.merge(
            unique_inetd,
            on=['actorname', 'objectname'],
            how='left',
            indicator=True
        )
        
        complement = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])
        print('Reduced Graph: ', len(complement))
        print()
    print('==========================================================')

In [ ]:
import networkx as nx

def trim_graph_from_outside(G1: nx.Graph, G2: nx.Graph, target: str):
    changed = True
    while changed:
        changed = False
        
        distances = nx.single_source_shortest_path_length(G1, target)
        if len(distances) == 1:
            break
        
        max_dist = max(distances.values())
        
        outer_layer = [n for n, dist in distances.items() 
                       if dist == max_dist and n != target]
        
        if not outer_layer:
            break
        
        for n in outer_layer:
            if n not in G2:
                continue
            
            G1_neighbors = set(G1.neighbors(n))
            G2_neighbors = set(G2.neighbors(n))
            
            if G1_neighbors == G2_neighbors:
                G1.remove_node(n)
                changed = True
            else:
                overlap_1hop = G1_neighbors.intersection(G2_neighbors)
                
                for neighbor in overlap_1hop:
                    if G1.has_edge(n, neighbor):
                        G1.remove_edge(n, neighbor)
                        changed = True
    return G1

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

from collections import deque, defaultdict

def k_hop_subgraph(edges, target_node, k):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)

    visited = {target_node}
    queue = deque([(target_node, 0)]) 
    while queue:
        node, depth = queue.popleft()
        if depth == k:
            continue
        for nbr in adj[node]:
            if nbr not in visited:
                visited.add(nbr)
                queue.append((nbr, depth + 1))

    sub_edges = [(u, v) for u, v in edges if u in visited and v in visited]
    return sub_edges

In [ ]:
def get_graph(df,target):
    raw_edges = df[['actorname', 'objectname']].values.tolist()
    edges =  k_hop_subgraph(raw_edges, target, 2)
    G = nx.Graph()
    G.add_edges_from(edges)
    return G

# Graph Reduction Results

In [ ]:
results = find_top_k_contributing_samples(all_score_matrices, k=1)
for malicious_proc, top_list in results.items():
    target_node = malicious_proc
    benign_labels = []

    print(f"Alert Entity : {malicious_proc}")
    for (omitted_sample, diff) in top_list:
        omitted_filename = omitted_sample.split('\\')[-1]
        benign_labels.append(omitted_filename)
    
    H1 = get_graph(test_df, target_node)
    print("Evaluation Graph:", len(H1.edges()))

    for bl in benign_labels:
        Hb = get_graph(train_df, bl)
        H1 = trim_graph_from_outside(H1, Hb, target_node)
        #print(f"{bl}: {len(H1.edges())} edges")
    print("Reduced Graph:", len(H1.edges()))
    print()

In [ ]:
benign_labels = []
results = find_top_k_contributing_samples(all_score_matrices, k=9)
for malicious_proc, top_list in results.items():
    if "procdump64.exe" in malicious_proc:
        print(f"Malicious process: {malicious_proc}")
        for (omitted_sample, diff) in top_list:
            omitted_filename = omitted_sample.split('\\')[-1]
            benign_labels.append(omitted_filename)
            print(f"  Omitted sample: {omitted_filename}, Score difference: {diff:.4f}")
        print()

In [ ]:
target_node = 'fsnotifier.exe'
H1 = get_graph(train_df, target_node)
print(len(H1.edges))
for bl in benign_labels:
    Hb = get_graph(train_df, bl)
    H1 = trim_graph_from_outside(H1, Hb, target_node)
    print(bl,len(H1.edges()))

In [ ]:
target_node = "git.exe"  
raw_edges = train_df[['actorname', 'objectname']].values.tolist()
edges =  k_hop_subgraph(raw_edges, target_node, 2)

In [ ]:
import networkx as nx
from pyvis.network import Network

# Build your main graph
G = nx.Graph()
G.add_edges_from(edges)

# Remove self loops (if any)
self_loops = list(nx.selfloop_edges(G))
G.remove_edges_from(self_loops)

# `nx.single_source_shortest_path_length` returns a dictionary: {node: distance}
distances = nx.single_source_shortest_path_length(G, target_node, cutoff=4)
# Extract all nodes (keys) found within distance <= 2
two_hop_nodes = list(distances.keys())

# Create a subgraph with only those nodes
H = G.subgraph(two_hop_nodes).copy()

# --- Now build a PyVis network from this subgraph ---
net = Network(
    notebook=True,
    height="750px",
    width="100%",
    bgcolor="#222222",
    font_color="white",
    cdn_resources='in_line'
)

# Add nodes with color customization
for node in H.nodes():
    if node == target_node:
        net.add_node(
            node,
            title=node,
            label=node,
            color="red"   # Highlight the target node in red
        )
    else:
        net.add_node(
            node,
            title=node,
            label=node,
            color="green" # Default color for non-target nodes
        )

# Add edges with color customization
for source, target in H.edges():
    # If the edge involves the target node, color it red
    if source == target_node or target == target_node:
        net.add_edge(source, target, color="red")
    else:
        net.add_edge(source, target, color="gray")

# (Optional) Customize layout/physics
net.set_options("""
var options = {
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -26,
            "centralGravity": 0.005,
            "springLength": 230,
            "springConstant": 0.18
        },
        "maxVelocity": 146,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {"iterations": 150}
    },
    "nodes": {
        "font": {
            "size": 12
        }
    }
}
""")

# Show or save
net.show("graph_explanation.html")

In [ ]:
import networkx as nx
from pyvis.network import Network

# Build your main graph
G = nx.Graph()
G.add_edges_from(edges)

# Remove self loops (if any)
self_loops = list(nx.selfloop_edges(G))
G.remove_edges_from(self_loops)

# `nx.single_source_shortest_path_length` returns a dictionary: {node: distance}
distances = nx.single_source_shortest_path_length(G, target_node, cutoff=4)
# Extract all nodes (keys) found within distance <= 2
two_hop_nodes = list(distances.keys())

# Create a subgraph with only those nodes
H = G.subgraph(two_hop_nodes).copy()

# --- Now build a PyVis network from this subgraph ---
net = Network(
    notebook=True,
    height="750px",
    width="100%",
    bgcolor="#222222",
    font_color="white",
    cdn_resources='in_line'
)

# Add nodes with color customization
for node in H.nodes():
    if node == target_node:
        net.add_node(
            node,
            title=node,
            label=node,
            color="red"   # Highlight the target node in red
        )
    else:
        net.add_node(
            node,
            title=node,
            label=node,
            color="green" # Default color for non-target nodes
        )

# Add edges with color customization
for source, target in H.edges():
    # If the edge involves the target node, color it red
    if source == target_node or target == target_node:
        net.add_edge(source, target, color="red")
    else:
        net.add_edge(source, target, color="gray")

# (Optional) Customize layout/physics
net.set_options("""
var options = {
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -26,
            "centralGravity": 0.005,
            "springLength": 230,
            "springConstant": 0.18
        },
        "maxVelocity": 146,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {"iterations": 150}
    },
    "nodes": {
        "font": {
            "size": 12
        }
    }
}
""")

# Show or save
net.show("graph_explanation.html")